# 🛡️ Real-Time Cybersecurity Log Monitoring & Alerting Dashboard
**Domain:** Cybersecurity, Data Analytics  
**Type:** Industry-Oriented Project  
**Objective:** Real-time threat detection using system logs

---
### 📌 What This Notebook Does:
1. **Generates / uploads** synthetic or real system log files
2. **Parses & normalizes** raw log data
3. **Detects threats** using rule-based logic (brute-force, suspicious IPs, etc.)
4. **Classifies severity** — Low / Medium / High
5. **Stores** logs and alerts in a local SQLite database
6. **Visualizes** everything in an interactive dashboard


## ⚙️ Section 1 — Install Dependencies

In [ ]:
!pip install pandas matplotlib seaborn plotly ipywidgets faker -q

## 📦 Section 2 — Imports

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import random
import re
import os
import datetime
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from faker import Faker
from collections import Counter
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

fake = Faker()
print('✅ All libraries loaded successfully.')

## 📝 Section 3 — Log Generation (Synthetic Data)
> If you have a real log file, skip this section and upload it in Section 4.

In [ ]:
# ─────────────────────────────────────────────
#  CONFIG — tweak these to control the dataset
# ─────────────────────────────────────────────
NUM_LOGS        = 500      # total log lines to generate
BRUTE_FORCE_IPS = 3        # IPs that will trigger brute-force patterns
ATTACK_RATIO    = 0.25     # fraction of logs that are attack-related

SUSPICIOUS_IPS = [fake.ipv4() for _ in range(BRUTE_FORCE_IPS)]

EVENT_TYPES = [
    ("Failed password",          "auth",    0.20),
    ("Accepted password",        "auth",    0.15),
    ("sudo command executed",    "sudo",    0.08),
    ("New connection",           "ssh",     0.12),
    ("Port scan detected",       "network", 0.05),
    ("Invalid user",             "auth",    0.10),
    ("Connection closed",        "ssh",     0.10),
    ("Firewall rule triggered",  "network", 0.07),
    ("Disk usage critical",      "system",  0.05),
    ("Service started",          "system",  0.08),
]

USERNAMES = ["root", "admin", "user", "ubuntu", "test", "guest", fake.user_name(), fake.user_name()]

def generate_logs(n=NUM_LOGS):
    logs = []
    base_time = datetime.datetime(2024, 6, 1, 0, 0, 0)
    for i in range(n):
        # pick event
        events, _, weights = zip(*EVENT_TYPES)
        event = random.choices(events, weights=weights, k=1)[0]

        # pick IP — bias suspicious IPs during attack events
        is_attack = event in ["Failed password", "Port scan detected", "Invalid user", "Firewall rule triggered"]
        if is_attack and random.random() < ATTACK_RATIO:
            ip = random.choice(SUSPICIOUS_IPS)
        else:
            ip = fake.ipv4()

        timestamp = base_time + datetime.timedelta(seconds=random.randint(0, 86400*7))
        user = random.choice(USERNAMES)
        port = random.choice([22, 80, 443, 3306, 8080, 21])
        log_category = [c for e, c, _ in EVENT_TYPES if e == event][0]

        raw = (f"{timestamp.strftime('%b %d %H:%M:%S')} server sshd[{random.randint(1000,9999)}]: "
               f"{event} for {user} from {ip} port {port} ssh2")

        logs.append({
            "timestamp": timestamp,
            "ip_address": ip,
            "username": user,
            "event": event,
            "category": log_category,
            "port": port,
            "raw_log": raw
        })
    return pd.DataFrame(logs).sort_values("timestamp").reset_index(drop=True)

df_logs = generate_logs()
print(f"✅ Generated {len(df_logs)} log entries")
print(f"🚨 Suspicious IPs injected: {SUSPICIOUS_IPS}")
df_logs.head(10)

## 📂 Section 4 — (Optional) Upload Your Own Log File
> Supported formats: plain `.log` or `.txt` (SSH / auth log style).  
> If you skip this, the synthetic data from Section 3 is used.

In [ ]:
USE_UPLOADED_FILE = False   # Set to True if you want to upload a real log file

if USE_UPLOADED_FILE:
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]

    LOG_PATTERN = re.compile(
        r'(?P<month>\w+)\s+(?P<day>\d+)\s+(?P<time>[\d:]+)\s+\S+\s+\S+:\s+'
        r'(?P<event>.+?)\s+for\s+(?P<user>\S+)\s+from\s+(?P<ip>[\d.]+)\s+port\s+(?P<port>\d+)'
    )

    rows = []
    with open(filename, 'r', errors='ignore') as f:
        for line in f:
            m = LOG_PATTERN.search(line)
            if m:
                rows.append({
                    "timestamp": pd.to_datetime(
                        f"2024 {m.group('month')} {m.group('day')} {m.group('time')}",
                        format="%Y %b %d %H:%M:%S", errors='coerce'),
                    "ip_address": m.group('ip'),
                    "username":   m.group('user'),
                    "event":      m.group('event').strip(),
                    "category":   "auth",
                    "port":       int(m.group('port')),
                    "raw_log":    line.strip()
                })

    df_logs = pd.DataFrame(rows).dropna(subset=['timestamp']).reset_index(drop=True)
    print(f"✅ Parsed {len(df_logs)} entries from '{filename}'")
else:
    print("ℹ️  Using synthetic log data (set USE_UPLOADED_FILE=True to upload your own).")

## 🔍 Section 5 — Threat Detection Engine
Rule-based detection:
- **Brute-force**: ≥5 failed logins from same IP within 10 minutes
- **Port scan**: IP triggers "Port scan detected" event
- **Root login attempt**: Failed login for root user
- **Suspicious firewall**: Firewall rule triggered
- **Invalid user**: Login attempt for non-existent user

In [ ]:
BRUTE_FORCE_THRESHOLD = 5   # failed attempts
BRUTE_FORCE_WINDOW    = 10  # minutes

def classify_severity(threat_type):
    severity_map = {
        "Brute-Force Attack":        "High",
        "Port Scan":                 "High",
        "Root Login Attempt":        "High",
        "Firewall Rule Triggered":   "Medium",
        "Invalid User Login":        "Medium",
        "Repeated Failed Login":     "Low",
    }
    return severity_map.get(threat_type, "Low")

def detect_threats(df):
    alerts = []

    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    # ── Rule 1: Brute-Force (≥5 failures in 10-min window per IP)
    failed = df[df['event'] == 'Failed password'].copy()
    failed = failed.sort_values('timestamp')
    for ip, grp in failed.groupby('ip_address'):
        grp = grp.reset_index(drop=True)
        for i in range(len(grp)):
            window = grp[(grp['timestamp'] >= grp.loc[i,'timestamp']) &
                         (grp['timestamp'] <= grp.loc[i,'timestamp'] + pd.Timedelta(minutes=BRUTE_FORCE_WINDOW))]
            if len(window) >= BRUTE_FORCE_THRESHOLD:
                threat = "Brute-Force Attack"
                alerts.append({
                    "timestamp":   grp.loc[i,'timestamp'],
                    "ip_address":  ip,
                    "username":    grp.loc[i,'username'],
                    "threat_type": threat,
                    "severity":    classify_severity(threat),
                    "detail":      f"{len(window)} failed logins in {BRUTE_FORCE_WINDOW} min"
                })
                break  # one alert per IP per window

    # ── Rule 2: Port Scan
    port_scans = df[df['event'] == 'Port scan detected']
    for _, row in port_scans.iterrows():
        threat = "Port Scan"
        alerts.append({
            "timestamp":   row['timestamp'],
            "ip_address":  row['ip_address'],
            "username":    row['username'],
            "threat_type": threat,
            "severity":    classify_severity(threat),
            "detail":      "Port scan activity detected from IP"
        })

    # ── Rule 3: Root Login Attempts
    root_attempts = df[(df['event'] == 'Failed password') & (df['username'] == 'root')]
    for _, row in root_attempts.iterrows():
        threat = "Root Login Attempt"
        alerts.append({
            "timestamp":   row['timestamp'],
            "ip_address":  row['ip_address'],
            "username":    "root",
            "threat_type": threat,
            "severity":    classify_severity(threat),
            "detail":      "Failed login attempt for root account"
        })

    # ── Rule 4: Firewall Rule Triggered
    fw_events = df[df['event'] == 'Firewall rule triggered']
    for _, row in fw_events.iterrows():
        threat = "Firewall Rule Triggered"
        alerts.append({
            "timestamp":   row['timestamp'],
            "ip_address":  row['ip_address'],
            "username":    row['username'],
            "threat_type": threat,
            "severity":    classify_severity(threat),
            "detail":      f"Firewall triggered on port {row['port']}"
        })

    # ── Rule 5: Invalid User
    invalid = df[df['event'] == 'Invalid user']
    for _, row in invalid.iterrows():
        threat = "Invalid User Login"
        alerts.append({
            "timestamp":   row['timestamp'],
            "ip_address":  row['ip_address'],
            "username":    row['username'],
            "threat_type": threat,
            "severity":    classify_severity(threat),
            "detail":      f"Login attempt for non-existent user '{row['username']}'"
        })

    df_alerts = pd.DataFrame(alerts)
    if not df_alerts.empty:
        df_alerts = df_alerts.drop_duplicates().sort_values('timestamp').reset_index(drop=True)
    return df_alerts

df_alerts = detect_threats(df_logs)
print(f"🚨 Total Alerts Generated: {len(df_alerts)}")
print(df_alerts['threat_type'].value_counts().to_string())
df_alerts.head(10)

## 🗄️ Section 6 — Store in SQLite Database

In [ ]:
DB_PATH = "log_monitor.db"

conn = sqlite3.connect(DB_PATH)

# Store logs
df_logs.to_sql("logs", conn, if_exists="replace", index=False)

# Store alerts
if not df_alerts.empty:
    df_alerts.to_sql("alerts", conn, if_exists="replace", index=False)

conn.commit()
print(f"✅ Database saved: {DB_PATH}")

# Quick query verification
print("\n📋 Sample query — High severity alerts:")
query_result = pd.read_sql(
    "SELECT timestamp, ip_address, threat_type, severity, detail FROM alerts WHERE severity='High' LIMIT 5",
    conn
)
display(query_result)
conn.close()

## 📊 Section 7 — Visualization Dashboard

In [ ]:
# ── KPI Summary Cards ──
total_logs   = len(df_logs)
total_alerts = len(df_alerts)
high_alerts  = len(df_alerts[df_alerts['severity'] == 'High'])  if not df_alerts.empty else 0
unique_ips   = df_logs['ip_address'].nunique()
top_attacker = df_alerts['ip_address'].mode()[0] if not df_alerts.empty else 'N/A'

summary_html = f"""
<div style='display:flex; gap:16px; font-family:monospace; flex-wrap:wrap;'>
  <div style='background:#1e2a3a;color:#00e5ff;padding:20px 30px;border-radius:10px;text-align:center;min-width:140px;'>
    <div style='font-size:36px;font-weight:bold;'>{total_logs}</div>
    <div style='font-size:13px;'>Total Logs</div>
  </div>
  <div style='background:#1e2a3a;color:#ff5252;padding:20px 30px;border-radius:10px;text-align:center;min-width:140px;'>
    <div style='font-size:36px;font-weight:bold;'>{total_alerts}</div>
    <div style='font-size:13px;'>Total Alerts</div>
  </div>
  <div style='background:#1e2a3a;color:#ff6d00;padding:20px 30px;border-radius:10px;text-align:center;min-width:140px;'>
    <div style='font-size:36px;font-weight:bold;'>{high_alerts}</div>
    <div style='font-size:13px;'>High Severity</div>
  </div>
  <div style='background:#1e2a3a;color:#69f0ae;padding:20px 30px;border-radius:10px;text-align:center;min-width:140px;'>
    <div style='font-size:36px;font-weight:bold;'>{unique_ips}</div>
    <div style='font-size:13px;'>Unique IPs</div>
  </div>
  <div style='background:#1e2a3a;color:#ea80fc;padding:20px 30px;border-radius:10px;text-align:center;min-width:200px;'>
    <div style='font-size:18px;font-weight:bold;'>{top_attacker}</div>
    <div style='font-size:13px;'>Top Attacker IP</div>
  </div>
</div>
"""
display(HTML("<h3>🔐 Security Dashboard — KPIs</h3>"))
display(HTML(summary_html))

In [ ]:
# ── Chart 1: Alert Severity Distribution ──
if not df_alerts.empty:
    severity_counts = df_alerts['severity'].value_counts().reset_index()
    severity_counts.columns = ['Severity', 'Count']
    color_map = {'High': '#ff5252', 'Medium': '#ff9800', 'Low': '#4caf50'}

    fig = px.pie(
        severity_counts, values='Count', names='Severity',
        title='🔴 Alert Severity Distribution',
        color='Severity', color_discrete_map=color_map,
        hole=0.45
    )
    fig.update_traces(textinfo='percent+label+value', pull=[0.05]*len(severity_counts))
    fig.update_layout(template='plotly_dark', height=400)
    fig.show()
else:
    print('⚠️ No alerts found — pie chart skipped.')

In [ ]:
# ── Chart 2: Threat Type Bar Chart ──
if not df_alerts.empty:
    threat_counts = df_alerts['threat_type'].value_counts().reset_index()
    threat_counts.columns = ['Threat Type', 'Count']

    fig2 = px.bar(
        threat_counts, x='Count', y='Threat Type',
        orientation='h',
        title='⚠️ Detected Threats by Type',
        color='Count',
        color_continuous_scale='Reds'
    )
    fig2.update_layout(template='plotly_dark', height=400, yaxis={'categoryorder': 'total ascending'})
    fig2.show()

In [ ]:
# ── Chart 3: Log Events Over Time ──
df_logs['hour'] = pd.to_datetime(df_logs['timestamp']).dt.floor('H')
time_series = df_logs.groupby(['hour', 'event']).size().reset_index(name='count')

# Focus on key events for readability
key_events = ['Failed password', 'Port scan detected', 'Invalid user', 'Firewall rule triggered']
ts_filtered = time_series[time_series['event'].isin(key_events)]

fig3 = px.line(
    ts_filtered, x='hour', y='count', color='event',
    title='📈 Security Events Over Time (Hourly)',
    markers=True
)
fig3.update_layout(template='plotly_dark', height=420, xaxis_title='Time', yaxis_title='Count')
fig3.show()

In [ ]:
# ── Chart 4: Top Attacker IPs ──
if not df_alerts.empty:
    top_ips = df_alerts['ip_address'].value_counts().head(10).reset_index()
    top_ips.columns = ['IP Address', 'Alert Count']

    fig4 = px.bar(
        top_ips, x='Alert Count', y='IP Address',
        orientation='h',
        title='🌐 Top 10 Attacker IP Addresses',
        color='Alert Count',
        color_continuous_scale='Oranges'
    )
    fig4.update_layout(template='plotly_dark', height=420, yaxis={'categoryorder': 'total ascending'})
    fig4.show()

In [ ]:
# ── Chart 5: Heatmap — Events by Hour of Day and Day of Week ──
df_logs['dow']  = pd.to_datetime(df_logs['timestamp']).dt.day_name()
df_logs['hour_of_day'] = pd.to_datetime(df_logs['timestamp']).dt.hour

heatmap_data = df_logs[df_logs['event'] == 'Failed password'].groupby(
    ['dow', 'hour_of_day']).size().unstack(fill_value=0)

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heatmap_data = heatmap_data.reindex([d for d in day_order if d in heatmap_data.index])

plt.figure(figsize=(16, 5))
sns.heatmap(heatmap_data, cmap='YlOrRd', linewidths=0.3, annot=False)
plt.title('🔥 Failed Login Heatmap — Hour of Day × Day of Week', fontsize=14, pad=12)
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 6: Log Category Breakdown ──
cat_counts = df_logs['category'].value_counts().reset_index()
cat_counts.columns = ['Category', 'Count']

fig6 = px.bar(
    cat_counts, x='Category', y='Count',
    title='📂 Log Events by Category',
    color='Count', color_continuous_scale='Blues'
)
fig6.update_layout(template='plotly_dark', height=380)
fig6.show()

## 🚨 Section 8 — Alert Table (Full Report)

In [ ]:
def severity_color(val):
    colors = {'High': 'background-color:#ff525240; color:#ff5252; font-weight:bold',
              'Medium': 'background-color:#ff980040; color:#ff9800; font-weight:bold',
              'Low': 'background-color:#4caf5040; color:#4caf50; font-weight:bold'}
    return colors.get(val, '')

if not df_alerts.empty:
    display(HTML('<h3>🗒️ All Security Alerts</h3>'))
    table = df_alerts[['timestamp','ip_address','username','threat_type','severity','detail']].copy()
    try:
        # pandas >= 2.1
        styled = table.style.map(severity_color, subset=['severity'])
    except AttributeError:
        # pandas < 2.1 fallback
        styled = table.style.applymap(severity_color, subset=['severity'])
    display(styled)
else:
    print('✅ No alerts detected — system is clean!')

## 💾 Section 9 — Export Results

In [ ]:
# Export logs and alerts to CSV
df_logs.to_csv("all_logs.csv", index=False)

if not df_alerts.empty:
    df_alerts.to_csv("security_alerts.csv", index=False)
    print("✅ Exported: all_logs.csv, security_alerts.csv")
else:
    print("✅ Exported: all_logs.csv (no alerts to export)")

# Download files in Colab
try:
    from google.colab import files
    files.download("all_logs.csv")
    if not df_alerts.empty:
        files.download("security_alerts.csv")
    files.download(DB_PATH)
    print("📥 Files downloaded to your local machine.")
except ImportError:
    print("ℹ️ Not running in Colab — files saved locally.")

---
## ✅ Summary

| Component | Status |
|---|---|
| Log Generation / Ingestion | ✅ Done |
| Parsing & Normalization | ✅ Done |
| Rule-Based Threat Detection | ✅ Done |
| Severity Classification | ✅ Done |
| SQLite Database Storage | ✅ Done |
| Interactive Dashboard | ✅ Done |
| CSV / DB Export | ✅ Done |

### 🔭 Future Scope (as per project)
- Add ML-based anomaly detection (Isolation Forest / Autoencoders)
- Support more log formats (Windows Event Logs, Apache, Nginx)
- Deploy on cloud with Docker
- Advanced alert correlation and reporting
